# CHIT No. 24 — Scrape Three Pages and Save CSV

Scrapes at least three pages of books.toscrape.com, converts the price to a numeric value, saves the dataframe as CSV and displays the average price.

**How to use:** Run each cell from top to bottom. Read the short explanation before every code cell.

### What this cell does
Import the libraries used for requests, HTML parsing and dataframe operations.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

### What this cell does
Set the website's first-page URL and create an empty list to collect all book records.

In [ ]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"
all_books = []

### What this cell does
Visit pages 1, 2 and 3. Extract title, price, rating and availability from each book card.

In [ ]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

for page_num in range(1, 4):
    url = base_url.format(page_num)
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):
        rating_word = book.select_one("p.star-rating")["class"][1]
        all_books.append({
            "Title": book.h3.a["title"],
            "Price": book.select_one(".price_color").get_text(strip=True),
            "Rating": rating_map.get(rating_word),
            "Availability": book.select_one(".availability").get_text(" ", strip=True),
            "Page": page_num
        })

    print(f"Page {page_num} scraped: {len(soup.select('article.product_pod'))} books")

### What this cell does
Convert the collected rows into a dataframe and check the first few records.

In [ ]:
df = pd.DataFrame(all_books)
df.head()

### What this cell does
Clean the Price column by removing the currency symbol and converting the remaining text to float.

In [ ]:
df["Price"] = (
    df["Price"]
    .str.replace("£", "", regex=False)
    .str.replace("Â", "", regex=False)
    .astype(float)
)

print(df.dtypes)
df.head()

### What this cell does
Save the cleaned dataframe to a CSV file and print the average price.

In [ ]:
csv_filename = "books_first_three_pages.csv"
df.to_csv(csv_filename, index=False)

print("Saved file:", csv_filename)
print("Total books scraped:", len(df))
print("Average book price: £", round(df["Price"].mean(), 2))